TOOLS - 

In [3]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("What is the 1+1?")
response

AIMessage(content='<think>\nOkay, the user asked "What is the 1+1?" Hmm, that\'s a straightforward math question. Let me think. The answer is obviously 2, right? But wait, maybe they\'re looking for something deeper. Sometimes people ask this question in different contexts, like in binary or in a riddle.\n\nFirst, I should confirm the basic answer. In standard arithmetic, 1 plus 1 equals 2. That\'s the most common interpretation. But maybe they want to know about different number systems. For example, in binary, 1+1 would be 10, which is 2 in decimal. Or in a modulo system, like mod 2, 1+1 would be 0. \n\nAlso, there\'s the philosophical angle. Some people use 1+1 as a metaphor for relationships or teamwork, but the user didn\'t indicate that. They just asked the question directly. \n\nI should also consider if there\'s a trick here. Sometimes people ask 1+1 as a joke, expecting a non-mathematical answer. Like "1+1=11" when you just concatenate the numbers, or "1+1= window," where the 

In [9]:
from langchain.tools import tool

@tool   #decorator to mark this function as a tool
def get_Weather(location: str) -> str:
    """Get the current weather of a location""" # importatn docstring to describe the tool and its parameters
    # Here you can integrate any weather API to get the weather information
    return f"The current weather in {location} is sunny with a temperature of 25°C."


model_with_tools = model.bind_tools([get_Weather])  # bind the tool to the model  


In [ ]:
response = model_with_tools.invoke("What is the current weather in New York?")  # Now the model can use the tool to get the weather information
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}, Arguments: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the current weather in New York. Let me check the tools available. There\'s a function called get_Weather that takes a location parameter. The required parameter is location, which should be a string. Since the user specified New York, I need to call this function with "New York" as the location. I\'ll make sure the arguments are correctly formatted as JSON within the tool_call tags.\n', 'tool_calls': [{'id': 'tzen78hnh', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_Weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 157, 'total_tokens': 269, 'completion_time': 0.152914695, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.007115439, 'prompt_tokens_details': None, 'queue_time': 0.170355795, 'total_time': 0.160030134}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'o

### tool execution loop
tool result as context to llm

In [16]:
# Step 1: Model generates tool calls
messages = [{ "role": "user", "content": "What is the current weather in New York?" }]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tool calls and append results to messages
for tool_call in ai_msg.tool_calls:
    tool_result = get_Weather.invoke(tool_call) # context from tool call # execute the tool call to get the result
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in New York is sunny with a temperature of 25°C. It looks like a great day to be outdoors—maybe bring some sunscreen! 🌞


In [17]:
messages

[{'role': 'user', 'content': 'What is the current weather in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the current weather in New York. I need to use the get_Weather function they provided. Let me check the function parameters. The function requires a "location" parameter, which is a string. Since the user specified New York, I should call the function with "New York" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'zw79a8nq6', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_Weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 157, 'total_tokens': 268, 'completion_time': 0.184221924, 'completion_tokens_details': {'reasoning_tokens': 85}, 'prompt_time': 0.006707084, 'prompt_tokens_details': None, 'queue_time': 0.022217475, 'total_time': 0.190929008}, 'model_name': 'qwen